In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

sns.set_theme(style="whitegrid",font_scale=1.05)

In [ ]:
df=pd.read_excel("BLC.xlsx")
df.head()

,S.No.,Country,City,Company,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,CONTACT,Google Map,Category,Reviews,Comments
0,1,India,Maharashtra,Vaishnavi Computer Institute Khapar,NaN,NaN,NaN,NaN,08805031107,https://maps.app.goo.gl/jzzzH8rttfqtUJBk7,Computer Training,4.9,59
1,2,India,Maharashtra,COACHING CENTRE,NaN,NaN,NaN,NaN,08806143746,https://maps.app.goo.gl/PQPnyoaDxfsvxuRt7,Education Center,5.0,1
2,3,India,Maharashtra,"SHELKE ENT CLINIC_DR. SHARAD SHELKE (MBBS, MS ...",NaN,NaN,NaN,NaN,09511985253,https://maps.app.goo.gl/uDWQYGVVt7Jwo3AY7,ENT Clinic,5.0,11
3,4,India,Gujarat,NARMAD ENT HOSPITAL,NaN,NaN,NaN,NaN,08200386101,https://maps.app.goo.gl/SXCo1UQSwXue8z4U7,ENT Hospital,4.9,130
4,5,India,Bangalore,"Dr Nadeem’s Bangalore ENT care centre, Seshadr...",NaN,NaN,NaN,NaN,08023463510,https://maps.app.goo.gl/5ZJHJkgy9LQRNW717,ENT Clinic,4.7,57


In [ ]:
df['Category'].unique()

array(['Computer Training', 'Education Center', 'ENT Clinic ',
       'ENT Hospital', 'Mobile Repairing', 'English Spoken',
       'Sports Club', 'ENT Clinic', 'Eye Care Clinic', 'Hospital',
       'Coaching Center', 'Eye Clinic', 'Personality Coaching Center',
       'Computer Repair Center', 'Electrician', 'Diagnostic Center',
       'Laptop Repairing', 'Bone Care', 'Toys Store', 'Competitive Exams',
       'Tution Center', 'Ultrasound', 'Optical', 'Eye Center',
       'Software Company', 'Tutoring Services', 'Dental Clinic',
       'ENT Center', 'Clinic', 'Eye Hospital', 'Car Wash',
       'Computer Taining', 'Eye Care', 'Coaching Classes', 'Life Coach',
       'Computer Repair', 'clinic', 'Computer store', 'Coaching center',
       'Car wash', 'Traffic Clinic', 'Medical Clinic', 'Teen Clinic',
       'Employment center', 'medical clinic', 'Computer training',
       'fitness', 'Fitness', 'ENT Specialist', 'Dental care',
       'Car Service', 'Tyre Shops', 'Grocery Store'], dtype=ob

In [ ]:
df["Category"] = df["Category"].str.strip().str.title().fillna("Unknown")

def map_to_standard_category(cat):
    cat_lower = str(cat).lower().strip()

    if 'repair' in cat_lower or 'mobile' in cat_lower or 'laptop' in cat_lower:
        return 'Repair Center'
    if 'computer' in cat_lower or 'education' in cat_lower or 'english' in cat_lower or 'software' in cat_lower:
        return 'Training Center'
    if 'coaching' in cat_lower or 'personality' in cat_lower or 'life' in cat_lower or 'competitive' in cat_lower or 'tutoring' in cat_lower or 'tution' in cat_lower:
        return 'Coaching Center'
    if 'ent' in cat_lower:
        return 'ENT Clinic'
    if 'eye' in cat_lower or 'optical' in cat_lower:
        return 'Eye Clinic'
    if 'dental' in cat_lower:
        return 'Dental Clinic'
    if 'hospital' in cat_lower or 'medical' in cat_lower or 'clinic' in cat_lower or 'traffic' in cat_lower or 'teen' in cat_lower or 'diagnostic' in cat_lower or 'ultrasound' in cat_lower or 'bone' in cat_lower:
        return 'Medical Clinic'
    if 'car wash' in cat_lower or 'car service' in cat_lower or 'tyre' in cat_lower:
        return 'Auto Service'
    if 'fitness' in cat_lower or 'gym' in cat_lower or 'sport' in cat_lower:
        return 'Fitness & Sports'
    if 'grocery' in cat_lower or 'kirana' in cat_lower or 'market' in cat_lower or 'bazar' in cat_lower or 'supermarket' in cat_lower:
        return 'Grocery Store'
    if 'electric' in cat_lower:
        return 'Electrician'
    return 'Other'

df['Standard_Category'] = df['Category'].apply(map_to_standard_category)
print(df['Standard_Category'].value_counts())

Standard_Category
Auto Service        53
Training Center     43
Coaching Center     40
Grocery Store       35
ENT Clinic          34
Medical Clinic      31
Repair Center       22
Fitness & Sports    19
Electrician         11
Eye Clinic          10
Other                1
Name: count, dtype: int64


In [ ]:
india_ranks = {
    'Training Center': 1,
    'Coaching Center': 2,
    'Medical Clinic': 3,
    'ENT Clinic': 4,
    'Eye Clinic': 5,
    'Repair Center': 6,
    'Fitness & Sports': 7,
    'Auto Service': 8,
    'Grocery Store': 9,
    'Electrician': 10,
    'Other': 11
}

foreign_ranks = {
    'Medical Clinic': 1,
    'ENT Clinic': 2,
    'Eye Clinic': 3,
    'Training Center': 4,
    'Coaching Center': 5,
    'Grocery Store': 6,
    'Fitness & Sports': 7,
    'Auto Service': 8,
    'Repair Center': 9,
    'Electrician': 10,
    'Other': 11
}

def assign_lead_score_and_rank(row):
    country = str(row['Country']).strip().lower()
    cat = row['Standard_Category']

    if 'india' in country:
        rank = india_ranks.get(cat, 13)
        if rank <= 5: return pd.Series([0.85, rank])
        elif rank <= 9: return pd.Series([0.55, rank])
        else: return pd.Series([0.15, rank])
    else:
        rank = foreign_ranks.get(cat, 13)
        if rank <= 3: return pd.Series([0.85, rank])
        elif rank <= 7: return pd.Series([0.55, rank])
        else: return pd.Series([0.15, rank])

df[['Conversion_Probability', 'Rank']] = df.apply(assign_lead_score_and_rank, axis=1)
print(df[['Country', 'Standard_Category', 'Rank', 'Conversion_Probability']].head(20))

   Country Standard_Category  Rank  Conversion_Probability
0    India   Training Center   1.0                    0.85
1    India   Training Center   1.0                    0.85
2    India        ENT Clinic   4.0                    0.85
3    India        ENT Clinic   4.0                    0.85
4    India        ENT Clinic   4.0                    0.85
5    India     Repair Center   6.0                    0.55
6    India   Training Center   1.0                    0.85
7    India   Training Center   1.0                    0.85
8    India  Fitness & Sports   7.0                    0.55
9    India        ENT Clinic   4.0                    0.85
10   India        ENT Clinic   4.0                    0.85
11   India        Eye Clinic   5.0                    0.85
12   India        Eye Clinic   5.0                    0.85
13   India   Training Center   1.0                    0.85
14   India    Medical Clinic   3.0                    0.85
15   India   Coaching Center   2.0                    0.

In [ ]:
df['Reviews'] = pd.to_numeric(df['Reviews'], errors='coerce').fillna(0.0)
df['Comments'] = pd.to_numeric(df['Comments'], errors='coerce').fillna(0.0)

df["Company_Name_Length"] = df["Company"].str.len()
df["Company_Word_Count"] = df["Company"].str.split().str.len()

category_counts_dict = df["Standard_Category"].value_counts().to_dict()
df["Category_Frequency"] = df["Standard_Category"].map(category_counts_dict)

city_counts_dict = df["City"].value_counts().to_dict()
df["City_Frequency"] = df["City"].map(city_counts_dict)

le_city = LabelEncoder()
df["City_Encoded"] = le_city.fit_transform(df["City"])

le_category = LabelEncoder()
df["Category_Encoded"] = le_category.fit_transform(df["Standard_Category"])

print("Features created!")
print(df[["Company_Name_Length", "Company_Word_Count", "Category_Frequency", "City_Frequency"]].head())

Features created!
   Company_Name_Length  Company_Word_Count  Category_Frequency  City_Frequency
0                   35                   4                  43              57
1                   15                   2                  43              57
2                   50                   8                  34              57
3                   19                   3                  34              57
4                   52                   7                  34              53


In [ ]:
city_data = df["City"].value_counts().reset_index()
city_data.columns = ["City", "Count"]
city_data = city_data.sort_values(by="Count", ascending=False).head(15)

fig1, ax1 = plt.subplots(figsize=(10, 6))
sns.barplot(data=city_data, x="Count", y="City", palette="crest", ax=ax1, edgecolor="black", linewidth=0.6)
ax1.set_title("Distribution of Businesses by City (Top 15)", fontsize=14, fontweight="bold", pad=15)
ax1.set_xlabel("Number of Businesses", fontsize=12, fontweight="bold")
ax1.set_ylabel("City", fontsize=12, fontweight="bold")
ax1.spines[['top', 'right']].set_visible(False)

for i, v in enumerate(city_data["Count"]):
    ax1.text(v + 0.3, i, str(v), va='center', fontsize=10, fontweight='bold')
plt.tight_layout()
plt.savefig("distribution_of_businesses_by_city.png")
plt.close()

category_counts = df["Standard_Category"].value_counts().reset_index()
category_counts.columns = ["Standard_Category", "Count"]
category_counts = category_counts.sort_values(by="Count", ascending=False)

fig2, ax2 = plt.subplots(figsize=(12, 6))
sns.barplot(data=category_counts, x="Count", y="Standard_Category", palette="viridis", ax=ax2, edgecolor="black", linewidth=0.6)
ax2.set_title("Business Categories by Total Count", fontsize=15, fontweight="bold", pad=15)
ax2.set_xlabel("Number of Businesses", fontsize=12, fontweight="bold")
ax2.set_ylabel("Business Category", fontsize=12, fontweight="bold")
ax2.spines[['top', 'right']].set_visible(False)

for i, v in enumerate(category_counts["Count"]):
    ax2.text(v + 0.3, i, str(v), va='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig("top_business_categories_by_count.png", dpi=300)
plt.close()

print("Graphs generated successfully with improved styling.")

/tmp/ipykernel_542/1587939684.py:6: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=city_data, x="Count", y="City", palette="crest", ax=ax1, edgecolor="black", linewidth=0.6)
/tmp/ipykernel_542/1587939684.py:23: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=category_counts, x="Count", y="Standard_Category", palette="viridis", ax=ax2, edgecolor="black", linewidth=0.6)


Graphs generated successfully with improved styling.


In [ ]:
feature_names = ["Company_Name_Length", "Company_Word_Count",
                 "Category_Frequency", "City_Frequency",
                 "City_Encoded", "Category_Encoded"]
mask = df['Group_Size'] >= 2
df_for_split = df[mask]
df_excluded = df[~mask]

X_split = df_for_split[feature_names]
y_split = df_for_split["Conversion_Probability"]
w_split = df_for_split['Sample_Weight']
strat_key = df_for_split['Country'] + "_" + df_for_split['Standard_Category']

X_train, X_test, y_train, y_test, w_train, w_test = train_test_split(
    X_split, y_split, w_split, test_size=0.2, random_state=42, stratify=strat_key
)

xgb_model = XGBRegressor(n_estimators=100, max_depth=4, random_state=42, learning_rate=0.1)
xgb_model.fit(X_train, y_train, sample_weight=w_train)

print(f"Train R²: {xgb_model.score(X_train, y_train):.4f}")
print(f"Test R²: {xgb_model.score(X_test, y_test):.4f}")
df["Predicted_Probability"] = xgb_model.predict(df[feature_names])


Train R²: 0.9824
Test R²: 0.9663


In [ ]:
category_rankings = df.groupby("Standard_Category")["Predicted_Probability"].mean().reset_index()
category_rankings = category_rankings.sort_values(by="Predicted_Probability", ascending=False)

fig3, ax3 = plt.subplots(figsize=(12, 6))
sns.barplot(data=category_rankings, x="Predicted_Probability", y="Standard_Category", palette="viridis", ax=ax3)
ax3.set_title("Business Categories by Predicted Conversion Probability")
ax3.set_xlabel("Average Predicted Probability")
ax3.set_ylabel("Business Category")
plt.tight_layout()
plt.savefig("top_business_categories_by_probability.png")
plt.close()

/tmp/ipykernel_542/4158210153.py:5: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=category_rankings, x="Predicted_Probability", y="Standard_Category", palette="viridis", ax=ax3)


In [ ]:
country_counts = df["Country"].value_counts().reset_index()
country_counts.columns = ["Country", "Count"]

fig4, ax4 = plt.subplots(figsize=(8, 5))
sns.barplot(data=country_counts, x="Count",y="Country",palette="coolwarm",ax=ax4)
ax4.set_title("Lead Volume by Country")
ax4.set_xlabel("Number of Businesses")
ax4.set_ylabel("Country")
plt.tight_layout()
plt.savefig("lead_volume_by_country.png")
plt.close()

/tmp/ipykernel_542/3357595080.py:5: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=country_counts, x="Count",y="Country",palette="coolwarm",ax=ax4)


In [ ]:
columns_to_export = ["S.No.", "Country", "City", "Company", "CONTACT", "Google Map", "Category",
                      "Standard_Category", "Rank", "Predicted_Probability"]

indian_df = df[df["Country"].str.lower().str.contains("india")].sort_values(by="Rank", ascending=True)[columns_to_export]
foreign_df = df[~df["Country"].str.lower().str.contains("india")].sort_values(by="Rank", ascending=True)[columns_to_export]

wb = openpyxl.Workbook()
wb.remove(wb.active)

sheets_config = {
    f"Indian Businesses ({len(indian_df)})": (indian_df, "1F4E78"),
    f"Foreign Businesses ({len(foreign_df)})": (foreign_df, "4D4D4D")
}

for sheet_name, (sub_df, header_color) in sheets_config.items():
    ws = wb.create_sheet(title=sheet_name)
    ws.views.sheetView[0].showGridLines = False

    headers = list(sub_df.columns)
    ws.append(headers)

    # ---- Styles ----
    header_font = Font(name="Calibri", size=11, bold=True, color="FFFFFF")
    header_fill = PatternFill(start_color=header_color, end_color=header_color, fill_type="solid")
    center_align = Alignment(horizontal="center", vertical="center", wrap_text=True)
    left_align = Alignment(horizontal="left", vertical="center")
    thin_border = Border(
        left=Side(style="thin", color="D9D9D9"), right=Side(style="thin", color="D9D9D9"),
        top=Side(style="thin", color="D9D9D9"), bottom=Side(style="thin", color="D9D9D9")
    )
    row_fill_even = PatternFill(start_color="F2F6FA", end_color="F2F6FA", fill_type="solid")
    row_fill_odd = PatternFill(start_color="FFFFFF", end_color="FFFFFF", fill_type="solid")


    for col_num, header in enumerate(headers, 1):
        cell = ws.cell(row=1, column=col_num)
        cell.font = header_font
        cell.fill = header_fill
        cell.alignment = center_align
        cell.border = thin_border
    ws.row_dimensions[1].height = 28

    for r_idx, row_data in enumerate(sub_df.values, 2):
        row_fill = row_fill_even if r_idx % 2 == 0 else row_fill_odd
        for c_idx, value in enumerate(row_data, 1):
            cell = ws.cell(row=r_idx, column=c_idx, value=value)
            cell.border = thin_border
            cell.font = Font(name="Calibri", size=10.5)
            cell.fill = row_fill

            current_header = headers[c_idx - 1]

            if current_header in ["S.No", "CONTACT", "Rank"]:
                cell.alignment = center_align
                if current_header == "CONTACT":
                    cell.number_format = "@"
            elif current_header == "Predicted_Probability":
                cell.alignment = center_align
                cell.number_format = "0.0%"
                if isinstance(value, (int, float)):
                    if value >= 0.70:
                        cell.font = Font(name="Calibri", size=10.5, bold=True, color="1E7C3A")
                    elif value <= 0.30:
                        cell.font = Font(name="Calibri", size=10.5, color="A6A6A6")
            elif current_header == "Google Map":
                cell.alignment = left_align
                if pd.notna(value) and str(value).startswith("http"):
                    cell.hyperlink = str(value)
                    cell.font = Font(name="Calibri", size=10.5, color="0563C1", underline="single")
            else:
                cell.alignment = left_align

        ws.row_dimensions[r_idx].height = 20
    for col in ws.columns:
        max_len = max(len(str(cell.value or "")) for cell in col)
        col_letter = get_column_letter(col[0].column)
        ws.column_dimensions[col_letter].width = max(max_len + 3, 12)

    ws.freeze_panes = "A2"
    ws.auto_filter.ref = ws.dimensions

wb.save("leads_prioritized_by_country.xlsx")

print("\n--- PROCESS COMPLETE ---")
print("Successfully generated 'leads_prioritized_by_country.xlsx' with Indian and Foreign sheets sorted by Rank!")


--- PROCESS COMPLETE ---
Successfully generated 'leads_prioritized_by_country.xlsx' with Indian and Foreign sheets sorted by Rank!
